# 🎬 Thử nghiệm ứng viên 3: RealBasicVSR (OpenMMLab / MMagic)

- **Mô hình:** RealBasicVSR (Recurrent Video Super-Resolution with Image Cleaning & SPyNet)
- **Tác giả:** Kelvin C.K. Chan et al. / OpenMMLab
- **Giấy phép:** Apache 2.0 (Thương mại & Nguồn mở tự do)
- **Đặc điểm:** Tận dụng thông tin tuần hoàn qua các khung hình lân cận (recurrent propagation) kết hợp căn chỉnh luồng quang học (SPyNet), giúp video ổn định cao theo thời gian, không nhấp nháy, tốc độ xử lý nhanh hơn nhiều so với diffusion.
- **Quy đổi tỉ lệ:** Model gốc có scale 4x. Theo đúng quy định của Phase 1, đầu ra sẽ được resize chuẩn xác xuống 2x (1440x2560) bằng thuật toán Lanczos.

In [ ]:
# [1] Kiểm tra GPU (Khuyên dùng Tesla T4 16GB VRAM)
!nvidia-smi

In [ ]:
# [2] Thiết lập thư mục và tải dự án Eco Video Upscaler
%cd /content
!rm -rf /content/upscale_video
!git clone https://github.com/mloc07/upscale_video.git
%cd /content/upscale_video

# Cài đặt OpenMMLab MMagic & dependencies
!pip install -q openmim
!mim install -q "mmengine>=0.8.0"
!mim install -q "mmcv>=2.0.1"
!pip install -q "mmagic>=1.0.0" rich opencv-python pynvml

In [ ]:
# [3] Khởi tạo mô hình RealBasicVSR và thực thi suy luận trên 3 clip chuẩn
import time, json, os, cv2, subprocess
from pathlib import Path
import torch
from mmagic.apis import MMagicInferencer

out_dir = Path("eval/results/realbasicvsr")
out_dir.mkdir(parents=True, exist_ok=True)

print("Đang nạp mô hình RealBasicVSR từ MMagic...")
inferencer = MMagicInferencer(model_name="real_basicvsr")

clips = ["clip_A", "clip_B", "clip_C"]
for c in clips:
    in_clip = f"eval/clips/{c}.mp4"
    raw_out = out_dir / f"{c}_raw4x.mp4"
    final_out = out_dir / f"{c}.mp4"

    print(f"\n=== Đang xử lý {c} với RealBasicVSR ===")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    inferencer.infer(video=in_clip, result_out_dir=str(out_dir))
    elapsed = time.perf_counter() - t0

    peak_vram = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0

    # Resize Lanczos từ 4x xuống 2x chuẩn 1440x2560
    # Tìm file video do MMagic xuất ra
    generated_vids = list(out_dir.glob(f"*{c}*.mp4"))
    if generated_vids:
        gen_vid = generated_vids[0]
        cmd = [
            "ffmpeg", "-y", "-i", str(gen_vid),
            "-vf", "scale=1440:2560:flags=lanczos",
            "-c:v", "libx264", "-crf", "17", "-preset", "fast",
            "-c:a", "copy", str(final_out)
        ]
        subprocess.run(cmd, check=True)

    metrics = {
        "clip_name": c,
        "model_name": "realbasicvsr",
        "input_resolution": "720x1280",
        "output_resolution": "1440x2560",
        "video_duration_sec": 6.0,
        "processing_time_sec": round(elapsed, 2),
        "sec_per_sec": round(elapsed / 6.0, 2),
        "fps": round(180 / elapsed, 2),
        "peak_vram_mb": round(peak_vram, 1)
    }
    with open(out_dir / f"metrics_{c}.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"-> Hoàn tất {c}: {elapsed:.2f}s ({elapsed/6.0:.2f} s/s, peak VRAM: {peak_vram:.1f} MB)")

In [ ]:
# [4] Kết xuất video so sánh chi tiết mặt (Face Zoom) và toàn khung (Full)
!python scripts/eval_compare.py --models realbasicvsr ccsr nomosuni

In [ ]:
# [5] Đóng gói kết quả và tải về máy tính
!zip -r realbasicvsr_eval_results.zip eval/results/realbasicvsr/ eval/comparisons/
from google.colab import files
files.download('realbasicvsr_eval_results.zip')